# 真实再扩散与编辑攻击闭环运行入口

该 Notebook 用于 Colab GPU 冷启动: 挂载 Google Drive、拉取仓库、读取前序 aligned rescoring 图像包和 threshold calibration 包、加载 SD3.5 Medium 相关攻击与检测 pipeline、生成真实 attacked image 文件、记录 source / attacked image digest、重跑攻击后 formal detection, 并把所有核对文件打包到 Google Drive。

运行依赖: 必须先完成 `aligned_rescoring_run.ipynb` 和 `threshold_calibration_run.ipynb`。本入口可与 `conventional_geometric_attack_evaluation_run.ipynb` 并行运行, 但 `dataset_level_quality_run.ipynb` 和 `pilot_paper_result_closure_run.ipynb` 必须等待本入口产物写入 Drive。

正式逻辑位于 `paper_workflow/colab_utils/real_attack_evaluation.py`, Notebook 只作为远程执行入口。

## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 SD3.5 Medium 与 DDIM attacker model 访问权限, 并配置 `HF_TOKEN`。
3. 确认 Google Drive 中已有 aligned rescoring 包, 默认查找 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `aligned_rescoring/aligned_rescoring_package_*.zip``。
4. 确认 Google Drive 中已有 threshold calibration 包, 默认查找 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `threshold_calibration/threshold_calibration_package_*.zip``。
5. 默认输出目录为 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `real_attack_evaluation``。
6. 本入口覆盖再扩散、再生成、全局编辑、局部编辑、视觉改写和自适应去水印类攻击; 常规失真、几何变换与 photometric 攻击由 `conventional_geometric_attack_evaluation_run.ipynb` 覆盖。
7. 本入口会先打包所有核对文件, 再执行断言, 便于失败时回传诊断结果。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "pilot_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
# 依赖诊断已收敛到 paper_workflow.colab_utils.dependency_check, 在仓库 checkout 后统一执行。


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="real_attack_evaluation",
)
print(paper_run_environment)

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report
dependency_report = build_notebook_dependency_report("real_attack_evaluation")
print(dependency_report)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
import os

from paper_workflow.colab_utils.real_attack_evaluation import run_default_real_attack_evaluation_from_drive_plan

real_attack_summary = run_default_real_attack_evaluation_from_drive_plan(
    root='.',
    aligned_rescoring_drive_dir=os.environ['SLM_WM_ALIGNED_RESCORING_DRIVE_DIR'],
    threshold_calibration_drive_dir=os.environ['SLM_WM_THRESHOLD_CALIBRATION_DRIVE_DIR'],
    require_threshold_package=True,
)
real_attack_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/real_attack_evaluation/real_attack_run_summary.json')
input_manifest_path = Path('outputs/real_attack_evaluation/real_attack_input_package_manifest.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else real_attack_summary)
if input_manifest_path.exists():
    print(input_manifest_path.read_text(encoding='utf-8'))


In [ ]:
from paper_workflow.notebook_utils.notebook_entrypoint import package_workflow_outputs

archive_record = package_workflow_outputs(root='.', workflow_name="real_attack_evaluation")
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/real_attack_evaluation').glob('*.json')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/real_attack_evaluation').glob('*.csv')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

records_path_text = real_attack_summary.get('output_records_path') or ''
formal_records_path_text = real_attack_summary.get('formal_records_path') or ''
if records_path_text:
    records_path = Path(records_path_text)
    if records_path.is_file():
        print(records_path.read_text(encoding='utf-8')[:4000])
if formal_records_path_text:
    formal_records_path = Path(formal_records_path_text)
    if formal_records_path.is_file():
        print(formal_records_path.read_text(encoding='utf-8')[:4000])

assert real_attack_summary['run_decision'] == 'pass', real_attack_summary
assert real_attack_summary['real_attacked_image_closed_loop_ready'] is True, real_attack_summary
assert real_attack_summary['regeneration_attack_gpu_validation_ready'] is True, real_attack_summary
assert real_attack_summary['attack_detection_rerun_ready'] is True, real_attack_summary
assert real_attack_summary['formal_attack_detection_ready'] is True, real_attack_summary
assert real_attack_summary['real_attacked_image_count'] > 0, real_attack_summary
